# A1.4 · Blast radius as a design metric

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Security of AI*

---

**Risk.** Blast radius is asserted in review, never measured.

**Control.** Make it a number that appears in the design review.

**This lab.** Put a blast-radius number in the design review.

| | |
|---|---|
| Open-source tooling | OpenFGA, SPIRE |
| Open-weight models | Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A1.4"))

"Reduce blast radius" is advice. A number is a design metric — it moves when you change the design, and you can put it in a review.

In [ ]:
from cybercommons import planes
W = planes.Tool

designs = {
 "one agent, all tools": planes.Manifest("mono", [
     W("read_file"),
     W("write_file",  writes=True, scope="project"),
     W("deploy_prod", writes=True, scope="org", reversible=False),
     W("rotate_secrets", writes=True, scope="org", reversible=False)], rung="L2.5"),
 "split by scope": planes.Manifest("reader+writer", [
     W("read_file"), W("write_file", writes=True, scope="project")], rung="L2.5"),
 "split + gate the org-wide tools": planes.Manifest("gated", [
     W("read_file"),
     W("write_file",  writes=True, scope="project"),
     W("deploy_prod", writes=True, scope="org", reversible=False),
     W("rotate_secrets", writes=True, scope="org", reversible=False)],
     approval_required={"deploy_prod", "rotate_secrets"}, rung="L2.5"),
}
for name, m in designs.items():
    b = m.blast_radius()
    print(f"{name:34s} blast={b['total']:4d}  {b['per_tool']}")

Splitting the agent and gating the wide tools reach a similar number by different means — and they fail differently. Splitting survives a compromised agent; gating survives a compromised *tool list* but not a compromised approver. Architecture is choosing which failure you prefer.

### Expect

The monolithic design scores far highest (two irreversible org-wide tools at double weight). Splitting drops it by an order of magnitude; gating drops the same two tools to zero.

### Your turn

Compute the blast radius of your largest production agent. If the number is over 40, name which single tool contributes most and what it would cost to gate it.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A1.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*